# Solar Filament Segmentation Challenge 2026 - reproduce the submission

This notebook regenerates the submitted predictions from the **test images** and the **trained weights**.

Pipeline: image -> 25 overlapping 512 px tiles (stride 384) -> U-Net/ResNet34 -> Gaussian-blended probability map
-> threshold 0.5 -> connected components (pieces < 120 px dropped; nearby pieces grouped with a closing of k=31,
pixels unchanged) -> COCO RLE -> `submission.csv`.

**Before you run:** (1) download the competition test images from the Kaggle *Data* tab and put them, flat, into `test_images/`;
(2) a GPU is recommended (Colab T4 is enough). All settings live in `src/filament/config.py`.

*Verified: run on a Colab T4 GPU (about 5 minutes for the 180 images) it regenerated the submitted file - 1565 of 1566 RLE strings identical, the remaining mask has IoU 0.9983 with the submitted one. See README, section "How to verify".*

In [ ]:
# 1) install dependencies (Colab: if pip asks to restart the runtime, restart it and run again)
%pip install -q -r requirements.txt

In [ ]:
# 2) paths and settings
import os, sys, urllib.request
sys.path.insert(0, os.path.abspath("src"))

IMAGES_DIR   = "test_images"                 # 180 test images, flat folder
WEIGHTS_PATH = "best_baseline_unet.pth"      # trained weights (about 98 MB)
WEIGHTS_URL  = ""                            # <- URL of the GitHub Release asset (filled in by the author)
OUT_CSV      = "submission_closing31B.csv"

assert os.path.isdir(IMAGES_DIR), "Put the Kaggle test images into test_images/ first"
if not os.path.exists(WEIGHTS_PATH):
    assert WEIGHTS_URL, "Download the weights from the GitHub Release page and place them next to this notebook"
    urllib.request.urlretrieve(WEIGHTS_URL, WEIGHTS_PATH)
print("images:", len(os.listdir(IMAGES_DIR)), "| weights:", round(os.path.getsize(WEIGHTS_PATH)/1e6, 1), "MB")

In [ ]:
# 3) load the model
import torch
from filament.model import load_checkpoint
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
model = load_checkpoint(WEIGHTS_PATH, device)

In [ ]:
# 4) FINAL submission (closing k=31 used only for grouping; public score 0.32)
from filament.pipeline import make_submission
report = make_submission(IMAGES_DIR, model, device, OUT_CSV, closing_kernel=31)
report

In [ ]:
# 5) (optional) BASELINE submission: plain connected components (public score 0.31)
# report_base = make_submission(IMAGES_DIR, model, device, "submission_baseline.csv", closing_kernel=None)
# report_base

In [ ]:
# 6) (optional) compare with the file that was uploaded to Kaggle
# !python scripts/compare_submissions.py submission/submission_closing31B.csv submission_closing31B.csv

### What to expect
* 180 test images, 2 of them without any predicted filament, about 8.7 filaments per image on average for the final file.
* The self-check dictionary must show `rows_failing_rle_roundtrip = 0` and `images_with_overlapping_masks = 0`.
* Small floating-point differences between GPUs can change a few boundary pixels; step 6 reports how many RLE strings are identical.